# kaggle model server

edit `MODEL_KEY` in the config cell and run the notebook top to bottom to bring that model up. to swap models mid-session, just change `MODEL_KEY` and re-run the config + run cells — swapping always stops whatever's currently up first, it never tries to run two models at once.

harness code (`harness.py`, `model_registry.py`) is pulled in, not pasted in — see the setup cell for the two ways to do that.

In [ ]:
# --- setup: pull in the shared harness + registry -----------------------

# option a: clone from your repo (edit the url below).
# rm -rf first so re-running this cell doesn't fail on an existing dir.
!rm -rf /kaggle/working/harness_src && git clone https://github.com/Meru143/kaggle-model-server.git /kaggle/working/harness_src
import sys
sys.path.append("/kaggle/working/harness_src")

# option b: attach harness.py + model_registry.py as a Kaggle Dataset instead —
# comment out option a above and uncomment this line instead:
# sys.path.append("/kaggle/input/TODO-your-dataset-slug")

!pip install -q huggingface_hub requests

In [ ]:
# re-running setup + this cell picks up freshly-pulled code without a
# kernel restart (python caches imports; plain re-import returns stale code)
import importlib, sys
if "harness" in sys.modules:
    try:
        sys.modules["harness"].stop()  # old module owns any running procs
    except Exception:
        pass
for _m in ("model_registry", "harness"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from model_registry import MODELS
from harness import run, stop, list_quants

list(MODELS.keys())

## config — change this line to swap models

In [ ]:
# keys are the full hugging face repo ids -- see MODELS above / the dropdown.
# checklist: 1. bartowski/gemma-4-12B-it-GGUF (mainline build)
#   2. prism-ml/Ternary-Bonsai-27B-gguf (prismml fork, t4 kernel smoke test)
#   3. bartowski/Qwen_Qwen3.6-35B-A3B-GGUF (dual-gpu)
MODEL_KEY = "bartowski/gemma-4-12B-it-GGUF"  # <- change this, then re-run

# any registry field can be overridden per-call -- no registry edits needed.
# the trycloudflare url is publicly reachable; add api_key="..." to gate it
url = run(MODEL_KEY, MODELS,
          quant="Q3_K_M",     # try a smaller quant -- no filename needed
          ctx=16384)          # more context; omit anything to keep registry defaults
# list_quants(MODEL_KEY, MODELS)   # <- see what quants this repo offers, with sizes
url

In [ ]:
# prefer clicking to editing? uncomment + run this instead of the config cell:
# !pip install -q gradio
# from control_panel import launch_panel
# launch_panel(auth=("joy", "CHANGE-ME"))  # share link is public -- change the password

call `stop()` in a new cell to tear the current model down cleanly (e.g. before ending the session, or before manually starting something outside the harness).

if a model fails to come up, the server log is at `/kaggle/tmp/llama-server.log` (`!tail -50 /kaggle/tmp/llama-server.log`); the tunnel log is `/kaggle/tmp/cloudflared.log`.